# Capability 5: Contextual follow-up questions by maintaining conversation history

7/7 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 5`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap05_contextual_followup.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: follow-up omits the zone -- should reuse the prior one

**✅ PASS**

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01a')
_ = ask('What about its EBITDA margin?', '01b')

01a
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

01b
USER: What about its EBITDA margin?

[intent=data_query | sub_agents=['structured']]

AGENT: The EBITDA margin for **North America in Q1 2024** was **31.3%** (computed from the structured KPI data).

If useful, I can compare this against the prior-year quarter or other zones for context.

## 02: follow-up omits the period -- should reuse the prior one

**✅ PASS**

In [ ]:
_ = ask("What was EMEA's revenue in Q3 2025?", '02a')
_ = ask('And the volume?', '02b')

02a
USER: What was EMEA's revenue in Q3 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA’s revenue in Q3 2025 was **$2,524 million** (USD million) per the structured KPI data.

Would you like to see how this compares quarter-over-quarter or year-over-year, or a breakdown by market/brand within EMEA?

02b
USER: And the volume?

[intent=data_query | sub_agents=['structured']]

AGENT: For EMEA in Q3 2025, the volume was **24,149.0 thousand hectoliters (k hL)**.

| Metric | Value |
|---|---|
| Volume (thousand hL) | 24,149.0 |

If you'd like, I can compare this to another quarter or break the volume down further within EMEA.

## 03: 'and last year?' style follow-up implying a period shift

**✅ PASS**

In [ ]:
_ = ask("What was South America's revenue in 2025?", '03a')
_ = ask('And in 2024?', '03b')

03a
USER: What was South America's revenue in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: South America’s revenue in 2025 was **$11,954 million USD** (i.e., ~$11.95 billion), based on summed quarterly KPI data.

If useful, I can break this down by major country or brand within the South America zone, or compare it against 2024.

03b
USER: And in 2024?

[intent=comparison | sub_agents=['structured']]

AGENT: **South America — revenue (USD m), 2024**

| Metric | Value |
| --- | --- |
| Revenue 2024 | **12,423.0 USD m** |
| Revenue 2025 | 11,954.0 USD m |
| YoY change | **−469.0 USD m** |

So for 2024, South America's revenue came in at **12,423.0 USD m** (quarterly grain, summed across 2024). For reference, the same measure for 2025 is 11,954.0 USD m, i.e. a decline of 469.0 USD m year over year.

Note: the evidence covers only the zone-level revenue aggregate for these two years — no sub-regional, brand-level, or quarterly breakdown was retrieved, so I can't attribute

## 04: pronoun follow-up ('it', 'that') resolved via context

**✅ PASS**

In [ ]:
_ = ask("What was Middle Americas' revenue in 2025?", '04a')
_ = ask('Is that up or down from 2024?', '04b')

04a
USER: What was Middle Americas' revenue in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: Middle Americas' revenue in 2025 was **$17,376 million USD** (approximately **$17.4 billion**), based on the structured KPI data.

04b
USER: Is that up or down from 2024?

[intent=comparison | sub_agents=['structured']]

AGENT: **Up.** Middle Americas revenue came in at **$17,376.0M in 2025** versus **$17,071.0M in 2024** — an increase of **+$305.0M**, or roughly **+1.8%** year over year.

| Metric | 2024 | 2025 | Change |
| --- | --- | --- | --- |
| Revenue (USD m) | 17,071.0 | 17,376.0 | +305.0 (up) |

Note: these figures are aggregated from quarterly records (quarters non-null), so they represent full-year totals. No document or web evidence was retrieved for this comparison — it rests entirely on the structured KPI data.

Follow-ups you might find useful:
- How does Middle Americas' 2025 growth compare with other zones (e.g. North America, EMEA, APAC)?
- Was the increase dr

## 05: follow-up asking for a related KPI without repeating the zone

**✅ PASS**

In [ ]:
_ = ask("What was Asia Pacific's volume in Q2 2025?", '05a')
_ = ask('What about revenue for the same period?', '05b')

05a
USER: What was Asia Pacific's volume in Q2 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: Asia Pacific's volume in Q2 2025 was **23,716 thousand hectoliters (k hL)**.

This figure comes directly from the quarterly KPI data for Asia Pacific in Q2 2025.

If you'd like more context, I can look up the prior quarter (Q1 2025) or the same quarter last year (Q2 2024) for a comparison — just let me know.

05b
USER: What about revenue for the same period?

[intent=data_query | sub_agents=['structured']]

AGENT: For Asia Pacific in **Q2 2025**, revenue was **$1,658 million (USD)**.

This is the confirmed figure for the same period per the structured KPI data. If you’d like, I can check how this compares to Q1 2025 or Q2 2024, or break the revenue down by country/brand within Asia Pacific.

## 06: 3-turn chain, each turn depends on the previous one's context

**✅ PASS**

In [ ]:
_ = ask("What was North America's revenue in 2025?", '06a')
_ = ask('Break that down by quarter.', '06b')
_ = ask('Which quarter was strongest?', '06c')

06a
USER: What was North America's revenue in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: North America’s revenue in 2025 was **$14,208 million USD**.

06b
USER: Break that down by quarter.

[intent=data_query | sub_agents=['structured']]

AGENT: Below is the North America revenue (USD million) breakdown for 2025 by quarter:

| Quarter | Revenue (USD million) |
| --- | --- |
| Q1 2025 | $3,364 |
| Q2 2025 | $3,844 |
| Q3 2025 | $3,765 |
| Q4 2025 | $3,235 |

Q2 was the strongest quarter, while Q4 was the weakest.

Would you like to see the year-over-year change for each quarter, or a breakdown by brand/segment within North America?

06c
USER: Which quarter was strongest?

[intent=data_query | sub_agents=['structured']]

AGENT: Based on the available data, **Q2 2025** was the strongest quarter for North America revenue, with **$3,844 million** in revenue. This is the top quarter returned from the 2025 data.

Would you like to see how the other quarters compare, or che

## 07: follow-up explicitly overriding the zone should NOT keep the old one

**✅ PASS**

In [ ]:
_ = ask("What was EMEA's revenue in 2025?", '07a')
_ = ask('Now show me North America instead.', '07b')

07a
USER: What was EMEA's revenue in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA's revenue in 2025 was **$9,502 million USD** (approximately **$9.5 billion**), based on quarterly data for the region.

If useful, I can break this down by quarter or compare it against 2024 performance.

07b
USER: Now show me North America instead.

[intent=data_query | sub_agents=['structured']]

AGENT: Here are the **North America** revenue figures for 2025 (USD millions):

| Period | Revenue (USD million) |
|---|---|
| Q1 2025 | $3,364 |
| Q2 2025 | $3,844 |
| Q3 2025 | $3,765 |
| Q4 2025 | $3,235 |

Q2 2025 was the strongest quarter, and Q4 2025 was the lowest.

Want to compare this quarterly trend to 2024, or break North America down by country/brand?